In [71]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from uncertainties import ufloat, covariance_matrix
from uncertainties import unumpy as unp

In [72]:
from functions import conversion, rotate_data

def theta(t, A, tau, w, phase, h):
    return A * np.exp(-t/tau) * np.sin(w*t + phase) + h

def lin(x, a, b):
    return a * x + b
    
def fit( t, xdata, ydata,name, p0: list = [180, 2000, 0.017, -1.5, 630], xlabel=False, cutoff: int = 0,cutoffend:  int = -1, zero_pos: tuple[float | int] | None = None,
        h1: list[ufloat] | None = None, h2: list[ufloat] | None = None, plot: bool = False):

    x, y, x0 = rotate_data(xdata, ydata, zero_pos=zero_pos)
    
    popt, pcov = curve_fit(theta, t[cutoff:cutoffend], x[cutoff:cutoffend], p0=p0) 

    if x0 != None:
        o = ufloat(popt[4], pcov[4,4]**0.5) - x0
        result = np.append(unp.uarray(popt, np.sqrt(np.diag(pcov))), o)
    else:
        result = unp.uarray(popt, np.sqrt(np.diag(pcov)))

    if plot:
        print('haha no plot')
            

    return result

In [73]:
def tag_it(x: ufloat, tag: str):
    return ufloat(x.n, x.s, tag=tag)

In [112]:
#new 2h meas Tn

#conversion
h1 = unp.uarray([-200,72], [5,5]) # coordinates in virt m 
h2 = unp.uarray([89,-74 ], [5,5])# virt m

conv = conversion(h1,h2)


cut1 = 1500
cut2 = -3000

wf2 = tag_it(fit(*np.loadtxt('m9_tos/m9_p02.txt', skiprows=2, unpack=True), 'M9.2 far II', cutoff=cut1, cutoffend=cut2)[2], tag='\omega_f')
print('T_far = ', 2 * np.pi / wf2)
wn2 = tag_it(fit(*np.loadtxt('m9_tos/m9_n2.txt', skiprows=2, unpack=True), 'M9.2 near II', cutoff=cut1, cutoffend=cut2)[2], tag='\omega_n')
print('T_near = ', 2 * np.pi / wn2)
#new 2h meas Tn

# G tos

# laser meas
L =  ufloat(4.321, 0.001, tag='L') #m
    
#measured:
M = ufloat(1.5,0.01, tag='M') 
#technical drawing
m = 0.028  #0.028 #kg tech drawing

#I = m*l**2/2 #kg m^2, using MIT estimated formula
l = ufloat((ufloat(0.12, 0.001) - ufloat(0.0171, 0.001)).n, (ufloat(0.12, 0.001) - ufloat(0.0171, 0.001)).s, tag='l')
#l=unp.uarray(0.12, 0.001)
rk = ufloat(0.0171, 0.00001, tag='radius of pendulum spheres')# radius of pendulum spheres
#I = 2*m *( 2/5 * rk**2 + (l/2)**2)  #improvement of I
I = m*l**2/2



#values:
rm = l/2
rM = ufloat(0.146,0.002, tag='r_M')
r = rM-rm

delta_w = tag_it(wn2**2 - wf2**2, tag='(\Delta\omega)^2')

#calc
Cn = -M*m*(3 * rm**2 * rM**2 -r**2 * rm * rM)/r**5
#Cn = 2 * M * m * rm * rM / r**3


#G = I * (wn2**2 - wf2**2) / Cn
G = I * delta_w / Cn

print('G=',G)

T_far =  378.15+/-0.04
T_near =  378.322+/-0.012
G= (6.5+/-1.8)e-11


In [115]:
print(I, delta_w, Cn)

0.000148+/-0.000004 (-2.5+/-0.6)e-07 -0.57+/-0.07


In [113]:
for (var, error) in G.error_components().items():
    print(f"${var.tag}$ & " + f"${var:L}$" + f" & {round((error / G.s)**2, 3)}\\\\")

$l$ & $0.1029 \pm 0.0014$ & 0.039\\
$r_M$ & $0.1460 \pm 0.0020$ & 0.116\\
$M$ & $1.500 \pm 0.010$ & 0.001\\
$(\Delta\omega)^2$ & $\left(-2.5 \pm 0.6\right) \times 10^{-7}$ & 0.844\\
